In [1]:
print('Alô!')

Alô!


In [1]:
# Mostrar o caminho do Java instalado e configurado

import os

print(os.environ.get("JAVA_HOME"))

None


In [2]:
# Import SparkSession
from pyspark.sql import SparkSession

# Create SparkSession
spark = SparkSession.builder \
      .master("local[1]") \
      .appName("Aula Pyspark Olist") \
      .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/11 12:04:48 WARN Utils: Your hostname, Tux, resolves to a loopback address: 127.0.1.1; using 192.168.0.218 instead (on interface wlo1)
26/07/11 12:04:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/11 12:04:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/11 12:04:50 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("OlistDataset") \
    .getOrCreate()

schema = StructType([

    StructField("index", IntegerType(), True),

    StructField("order_id", StringType(), True),
    StructField("order_item_id", IntegerType(), True),

    StructField("customer_id", StringType(), True),
    StructField("customer_unique_id", StringType(), True),

    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True),

    StructField("product_id", StringType(), True),
    StructField("product_category_name", StringType(), True),

    StructField("product_name_lenght", DoubleType(), True),
    StructField("product_description_lenght", DoubleType(), True),
    StructField("product_photos_qty", DoubleType(), True),

    StructField("product_weight_g", DoubleType(), True),
    StructField("product_length_cm", DoubleType(), True),
    StructField("product_height_cm", DoubleType(), True),
    StructField("product_width_cm", DoubleType(), True),

    StructField("seller_id", StringType(), True),
    StructField("seller_city", StringType(), True),
    StructField("seller_state", StringType(), True),
    StructField("seller_zip_code_prefix", IntegerType(), True),

    StructField("payment_type", StringType(), True),
    StructField("payment_sequential", IntegerType(), True),
    StructField("payment_installments", IntegerType(), True),

    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True),
    StructField("payment_value", DoubleType(), True),

    StructField("shipping_limit_date", TimestampType(), True),
    StructField("order_purchase_timestamp", TimestampType(), True),
    StructField("order_approved_at", TimestampType(), True),
    StructField("order_delivered_carrier_date", TimestampType(), True),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True),

    StructField("day_of_purchase", StringType(), True),
    StructField("month_of_purchase", StringType(), True),
    StructField("year_of_purchase", IntegerType(), True),
    StructField("month/year_of_purchase", StringType(), True),

    StructField("order_status", StringType(), True),
    StructField("order_unique_id", StringType(), True)

])

# Fazer download do dataset
# https://www.kaggle.com/datasets/enzoschitini/brazilian-e-commerce-public-dataset-by-olist/data?select=Brazilian+E-Commerce+Public+Dataset+by+Olist.csv
# Renomear o arquivo para olist.csv

df = spark.read \
    .option("header", "true") \
    .option("nullValue", "") \
    .schema(schema) \
    .csv("datasets/olist/olist.csv")

df.show(5, truncate=False)

df.printSchema()

26/07/11 12:05:03 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
26/07/11 12:05:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/07/11 12:05:06 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , order_id, order_item_id, customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state, product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm, seller_id, seller_city, seller_state, seller_zip_code_prefix, payment_type, payment_sequential, payment_installments, price, freight_value, payment_value, shipping_limit_date, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_deliv

+-----+--------------------------------+-------------+--------------------------------+--------------------------------+------------------------+---------------------+--------------+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+--------------------------------+-------------+------------+----------------------+------------+------------------+--------------------+-----+-------------+-------------+-------------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+---------------+-----------------+----------------+----------------------+------------+----------------------------------+
|index|order_id                        |order_item_id|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_stat

In [4]:
df.createOrReplaceTempView("olist")

### Total de registros no dataset

In [5]:
spark.sql("""
    SELECT COUNT(*) AS total_registros
    FROM olist
""").show()

+---------------+
|total_registros|
+---------------+
|         113390|
+---------------+



### Verificar valores nulos

In [6]:
spark.sql("""
SELECT
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS order_id_nulos,
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS customer_id_nulos,
    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS product_id_nulos,
    SUM(CASE WHEN seller_id IS NULL THEN 1 ELSE 0 END) AS seller_id_nulos,
    SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS price_nulos,
    SUM(CASE WHEN freight_value IS NULL THEN 1 ELSE 0 END) AS freight_nulos,
    SUM(CASE WHEN payment_value IS NULL THEN 1 ELSE 0 END) AS payment_nulos
FROM olist
""").show(truncate=False)

+--------------+-----------------+----------------+---------------+-----------+-------------+-------------+
|order_id_nulos|customer_id_nulos|product_id_nulos|seller_id_nulos|price_nulos|freight_nulos|payment_nulos|
+--------------+-----------------+----------------+---------------+-----------+-------------+-------------+
|0             |0                |0               |0              |0          |0            |0            |
+--------------+-----------------+----------------+---------------+-----------+-------------+-------------+



In [7]:
spark.sql("""
SELECT
    SUM(CASE WHEN order_approved_at IS NULL THEN 1 ELSE 0 END) AS aprovacao_nula,
    SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS entrega_nula,
    SUM(CASE WHEN order_estimated_delivery_date IS NULL THEN 1 ELSE 0 END) AS previsao_nula
FROM olist
""").show()

+--------------+------------+-------------+
|aprovacao_nula|entrega_nula|previsao_nula|
+--------------+------------+-------------+
|             0|           0|            0|
+--------------+------------+-------------+



### Remover valores duplicados

In [8]:
spark.sql("""
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT order_unique_id) AS registros_unicos
FROM olist
""").show()

+---------------+----------------+
|total_registros|registros_unicos|
+---------------+----------------+
|         113390|          108640|
+---------------+----------------+



In [9]:
# Depois criar um novo dataframe sem duplicados

spark.sql("""
CREATE OR REPLACE TEMP VIEW olist_sem_duplicados AS
SELECT DISTINCT *
FROM olist
""")

DataFrame[]

### Converter todas as datas para timestamp

In [10]:
spark.sql("""
SELECT
    CAST(order_purchase_timestamp AS TIMESTAMP) AS order_purchase_timestamp,
    CAST(order_approved_at AS TIMESTAMP) AS order_approved_at,
    CAST(order_delivered_carrier_date AS TIMESTAMP) AS order_delivered_carrier_date,
    CAST(order_delivered_customer_date AS TIMESTAMP) AS order_delivered_customer_date,
    CAST(order_estimated_delivery_date AS TIMESTAMP) AS order_estimated_delivery_date,
    CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_date
FROM olist
""").show(5, truncate=False)

+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------------+
|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|shipping_limit_date|
+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------------+
|2017-09-13 08:59:02     |2017-09-13 09:45:35|2017-09-19 18:34:16         |2017-09-20 23:43:48          |2017-09-29 00:00:00          |2017-09-19 09:45:35|
|2017-06-28 11:52:20     |2017-06-29 02:44:11|2017-07-05 12:00:33         |2017-07-13 20:39:29          |2017-07-26 00:00:00          |2017-07-05 02:44:11|
|2018-05-18 10:25:53     |2018-05-18 12:31:43|2018-05-23 14:05:00         |2018-06-04 18:34:26          |2018-06-07 00:00:00          |2018-05-23 10:56:25|
|2017-08-01 18:38:42     |2017-08-01 18:55:08|2017-08-02 19:07:3

### Verificar preços negativos

In [11]:
spark.sql("""
SELECT *
FROM olist
WHERE price < 0
""").show()

+-----+--------+-------------+-----------+------------------+------------------------+-------------+--------------+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+---------+-----------+------------+----------------------+------------+------------------+--------------------+-----+-------------+-------------+-------------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+---------------+-----------------+----------------+----------------------+------------+---------------+
|index|order_id|order_item_id|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|seller_id|seller_city|seller_st